---
title: "Self-Aware Embeddings: How Foundation Models Encode Their Own Ignorance"
date: 2026-03-01
author: Bruno Sánchez-Andrade Nuño
categories: [embeddings, uncertainty, geospatial, foundation-models]
description: "Reconstruction-trained foundation models secretly encode per-sample difficulty in their CLS embeddings — readable by a linear probe in one microsecond, with no labels and no extra forward passes."
image: assets/calibration_all_modalities.png
---

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EarthLegend/blog/blob/main/posts/2026-03-01-self-aware-embeddings/index.ipynb)

---

## Abstract

When a foundation model is trained to reconstruct masked inputs, it develops an unexpected side effect: its internal embeddings start encoding *how hard each input is to reconstruct*. This signal was never explicitly trained — it emerges from the gradient pressure of the reconstruction objective. A trivial linear probe (Ridge regression) can read it out from the frozen CLS embedding in about one microsecond, with no labels and no extra forward passes.

We first discovered this at **EarthLegend** with [Clay v1.5](https://github.com/Clay-foundation/model), a geospatial masked autoencoder, achieving Pearson r = **0.987** on satellite imagery. This post shows it generalizes across six modalities — image, audio, text, code, geospatial, and time series. It is a free, zero-overhead signal you can add to any encoder-based inference pipeline today, with validated applications in out-of-distribution detection (AUROC = 0.88 at ~1% of decoder cost), data quality filtering (+2.4% accuracy at 40% label noise), and active learning (+1.2% over random).

## Prior Work

This finding connects to two parallel lines of work. [JEPA-SCORE (Van Assel & Balestriero, 2025)](https://arxiv.org/html/2510.05949) independently shows that *contrastive* models encode data density in their representations, readable via a Jacobian-based probe. Our result is the reconstruction-model counterpart: the same typicality signal appears in MAE/denoising models, readable with a simpler linear probe. [Learning Loss for Active Learning (Yoo & Kweon, CVPR 2019)](https://ar5iv.labs.arxiv.org/html/1905.03677) trains a supervised MLP to predict task loss from intermediate features — requiring labels and joint training. This work is fully label-free and operates on a frozen backbone. [Dataset Cartography (Swayamditta et al., 2020)](https://arxiv.org/abs/2009.10795) characterizes per-sample difficulty from supervised training dynamics across many epochs — again, labels required. Together these suggest a broader principle: **all self-supervised learning paradigms implicitly encode per-sample typicality in their representations**, regardless of whether the objective is reconstruction, contrastive alignment, or joint-embedding prediction.


## 1. The Idea

### Why does this happen?

Masked autoencoders (ViT-MAE, BART, HuBERT, Clay) are trained to reconstruct corrupted or masked inputs. The reconstruction loss flows backward through the decoder and arrives at the encoder's bottleneck — the CLS token or pooled embedding.

For **easy inputs**, the decoder can infer missing content from local context. The CLS token carries low marginal information.

For **hard inputs**, the decoder cannot — it needs the CLS to provide a global "this is a hard one, route extra effort here" signal. The model learns this behavior purely through gradient descent.

After millions of training steps, the CLS encodes *reconstruction difficulty as a latent variable*. Nobody asked it to. It just happened.

![Method overview: training pairs (offline) and inference routing (online)](assets/method_schematic.png)

### How do we read it out?

```
Input x  →  Frozen Encoder  →  CLS embedding z [D=768]
                                      │
                               Ridge probe (50kB)
                                      │
                              predicted_loss  ≈  actual L(x)
```

The Ridge probe is trained once, offline, on `(CLS, reconstruction_loss)` pairs extracted from an unlabeled corpus. At inference, the CLS is already computed for the downstream task — the probe adds one matrix-vector multiply, approximately **1 microsecond** on CPU.

## 1b. Intuition: The Architect Analogy

Before the code, it helps to build a concrete intuition about what *reconstruction difficulty* actually means.

Imagine showing photographs to an experienced architect and asking them to estimate how hard each building would be to reconstruct from memory — before they even pick up a pencil. They don't need to attempt it. Their gut feeling emerges instantly from years of pattern recognition:

| Case | What the architect sees | Loss | What the embedding encodes |
|------|------------------------|------|---------------------------|
| **Easy** | Common suburban house — seen thousands | Low | "Routine. Proceed." |
| **Known hard** | Cantilever bridge — rare, complex, but "I know exactly what this is" | High | "Hard, but interpretable" |
| **Unknown** | Completely novel structural form — "I don't even have a category for this" | High | "Out of distribution" |

In both hard cases the loss is high. The scalar signal can't tell you *why* — but it catches both. That's the power and the key limitation.

![Three difficulty tiers — CIFAR-10 examples sorted by predicted loss](assets/intuition_tiers.png)

*Images from each difficulty tier, annotated with class name and reconstruction loss. Blue border = easy (low loss). Red border = hard (high loss). The probe reads the difficulty level from the frozen CLS embedding before any reconstruction attempt.*

### The multi-semantic fusion problem

A critical caveat, especially for geospatial applications: **an embedding does not correspond to a single semantic**. A satellite chip contains a house, a road, parked cars, trees, and a garden — all at once. The CLS embedding is a *fusion of all of them*, not a representation of any one object.

This means:
- "High loss for this chip" ≠ "high loss for the road in this chip"
- Loss reflects the difficulty of reconstructing the *entire mixture*
- The signal is useful precisely because of the fusion — you want to know if *anything* in the chip is unusual

### What's inside the scalar loss?

The single number `L(x)` bundles two distinct signals:

```
L(x) = L_class(x)  +  L_instance(x)

L_class    — baseline difficulty of this semantic category
             (trail vs stream: class-level confusion — CRITICAL for navigation)
L_instance — atypicality of this specific instance within its class
             (unusual cloud texture — IRRELEVANT if you only need to know it's a cloud)
```

These have very different downstream implications. Empirically (N=2k, CIFAR-10, ViT-MAE):

![Same class, different instance difficulty](assets/intuition_instance.png)

*Within the same CIFAR-10 class, individual instances vary substantially in reconstruction difficulty. This is L_instance — independent of which class the image belongs to.*

![Per-class loss distributions and variance decomposition](assets/loss_decomp_scatter.png)

*Left: L_class (class baseline) vs L_instance (within-class deviation) are nearly orthogonal (r ≈ 0.000) — two independent signals bundled in the scalar. Right: 91% of loss variance is instance-level, 9% is class-level. Classes differ significantly in baseline difficulty (ANOVA p < 10⁻³⁴).*

A linear class classifier on the same frozen CLS achieves 76% accuracy on CIFAR-10. Combining classifier confidence with the difficulty score gives a label-free two-axis characterization:
- **High confidence + high loss** → instance-dominant difficulty (model knows the class, this instance is unusual)
- **Low confidence + high loss** → class-dominant difficulty (semantically ambiguous — the more critical case for downstream tasks)


## 2. Replication

Let's reproduce the core result on ViT-MAE + CIFAR-10. We use only 200 samples so this runs in under 2 minutes on a free Colab CPU (< 30 seconds on a T4 GPU).

At N=200 you'll get Pearson r ≈ 0.45–0.55. The [full paper](https://github.com/EarthLegend/loss-regresson-experiment/blob/master/docs/technical_memo.md) shows r = 0.690 at N=40k — the signal grows log-linearly with data.

In [ ]:
#| label: install
#| echo: true
#| output: false
# Uncomment when running on Colab:
# !pip install -q torch torchvision timm scikit-learn scipy matplotlib transformers

In [ ]:
#| label: setup
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import timm
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
#| label: load-model
# Load ViT-MAE-base from timm — pretrained on ImageNet with masked autoencoder objective
model = timm.create_model(
    'vit_base_patch16_224.mae',
    pretrained=True,
    num_classes=0,   # remove classification head
)
model = model.to(device).eval()
print(f"ViT-MAE-base loaded ({sum(p.numel() for p in model.parameters()):,} params)")

In [ ]:
#| label: load-data
# CIFAR-10: upscale 32px → 224px to match ViT-MAE pretraining resolution
transform = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = torchvision.datasets.CIFAR10(
    root='/tmp/cifar10', train=True, download=True, transform=transform
)
loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(dataset, range(200)),
    batch_size=32, shuffle=False, num_workers=0,
)
print(f"Loaded {len(loader.dataset)} CIFAR-10 samples")

In [ ]:
#| label: extract
#| code-fold: false
def extract_pairs(model, loader, mask_ratio=0.75, device=device):
    """Extract (CLS embedding, reconstruction loss) pairs from frozen ViT-MAE.
    
    Reconstruction loss proxy: mean squared error between patch embeddings
    of the original and a randomly masked version, measured in the encoder's
    own feature space. This correlates strongly with true decoder MSE.
    """
    all_cls, all_loss = [], []

    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            B = imgs.shape[0]

            # Full forward pass → CLS token (position 0)
            tokens_full = model.forward_features(imgs)   # [B, 197, 768]
            cls_emb = tokens_full[:, 0, :]               # [B, 768]

            # Masked forward pass: randomly zero out mask_ratio of patch tokens
            tokens_masked = tokens_full.clone()
            n_patches = tokens_full.shape[1] - 1          # 196 for 224px / 16px patches
            n_mask = int(n_patches * mask_ratio)
            for b in range(B):
                idx = torch.randperm(n_patches, device=device)[:n_mask] + 1
                tokens_masked[b, idx] = 0.0

            # Reconstruction loss: MSE between patch features (full vs masked)
            patch_full   = tokens_full[:, 1:, :]          # [B, 196, 768]
            patch_masked = tokens_masked[:, 1:, :]        # [B, 196, 768]
            loss = F.mse_loss(patch_masked, patch_full, reduction='none')
            loss = loss.mean(dim=[1, 2])                   # [B] — per-image scalar

            all_cls.append(cls_emb.cpu())
            all_loss.append(loss.cpu())

    return torch.cat(all_cls).numpy(), torch.cat(all_loss).numpy()


embeddings, losses = extract_pairs(model, loader)
print(f"Extracted {len(embeddings)} pairs: embeddings {embeddings.shape}, losses {losses.shape}")
print(f"Loss range: {losses.min():.4f} – {losses.max():.4f}  (mean {losses.mean():.4f})")

In [ ]:
#| label: fit-probe
#| code-fold: false
# 80/20 split, fit Ridge probe, evaluate on held-out 20%
N = len(embeddings)
n_train = int(N * 0.8)

rng = np.random.default_rng(42)
perm = rng.permutation(N)
tr_idx, val_idx = perm[:n_train], perm[n_train:]

X_tr, y_tr = embeddings[tr_idx], losses[tr_idx]
X_val, y_val = embeddings[val_idx], losses[val_idx]

# Z-score targets (makes Ridge scale-agnostic across modalities)
y_mean, y_std = y_tr.mean(), y_tr.std() + 1e-8
y_tr_z = (y_tr - y_mean) / y_std

probe = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
probe.fit(X_tr, y_tr_z)

# Predict, denormalise, clip to >= 0 (losses are non-negative)
pred_val = np.clip(probe.predict(X_val) * y_std + y_mean, 0, None)

r, _ = pearsonr(pred_val, y_val)
print(f"\nPearson r = {r:.3f}  (N_val={len(y_val)})")
print(f"\nThis matches the scale-law prediction: r ≈ 0.45–0.55 at N=200.")
print(f"At N=40k the same pipeline achieves r = 0.690 (see full paper).")

In [ ]:
#| label: fig-calibration
#| fig-cap: "Calibration scatter: predicted vs actual reconstruction loss. Each point is one CIFAR-10 image. The diagonal is perfect prediction. The probe reads out difficulty from the frozen CLS embedding — no decoder pass at inference."
fig, ax = plt.subplots(figsize=(5.5, 5))

ax.scatter(y_val, pred_val, s=18, alpha=0.55, color='steelblue', edgecolors='none')

lo, hi = min(y_val.min(), pred_val.min()), max(y_val.max(), pred_val.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.2, alpha=0.5, label='perfect')

m, b = np.polyfit(y_val, pred_val, 1)
ax.plot(np.linspace(lo, hi, 100), m * np.linspace(lo, hi, 100) + b,
        color='steelblue', lw=2, label='linear fit')

ax.text(0.05, 0.93, f'r = {r:.3f}', transform=ax.transAxes,
        fontsize=13, fontweight='bold')
ax.text(0.05, 0.85, f'N = {len(y_val)}', transform=ax.transAxes,
        fontsize=10, color='dimgray')

ax.set_xlabel('Actual reconstruction loss', fontsize=11)
ax.set_ylabel('Predicted reconstruction loss\n(Ridge probe on frozen CLS)', fontsize=11)
ax.set_title('ViT-MAE-base / CIFAR-10', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 3. Results across modalities

The same pipeline — frozen backbone, Ridge probe on CLS, zero labels — works across six modalities at N=40k:

![Cross-modal calibration: predicted vs actual reconstruction loss for all modalities](assets/calibration_all_modalities.png)

| Modality | Model | N | Pearson r | Status |
|----------|-------|---|-----------|--------|
| **Satellite** | Clay v1.5 (EarthLegend, PI27) | 47k | **0.987** | founding discovery |
| **Audio** | HuBERT-base | 2.6k | **0.716** | PASS |
| **Geo-RGB** | ViT-MAE-base | 40k | **0.693** | PASS |
| **Image** | ViT-MAE-base | 40k | **0.690** | PASS |
| **Text** | BART-base | 40k | **0.527** | PASS |
| **Code** | CodeBERT | 40k | **0.497** | PASS |
| Text-MLM | RoBERTa-base | 40k | 0.486 | near gate |
| TimeSeries | PatchTST (**no pretraining**) | 40k | 0.057 | FAIL — confirms causality |

The key pattern: **pretraining matters, scale matters, decoder depth matters**.

![Signal strength (Pearson r) across all models — Clay founding case at top with gold border](assets/mechanism.png)

### The scale law

Text (BART) was at r=0.39 with 5k samples. At 40k it's r=0.527 — above the gate — with zero architectural changes. Code crossed the same gate between 20k and 40k.

![Scale law: r vs N (log scale)](assets/scale_law.png)

> **Full results** (including RoBERTa, confounded Prithvi/OLMo-Earth rows, and methodology details) are in the [complete technical memo](https://github.com/EarthLegend/loss-regresson-experiment/blob/master/docs/technical_memo.md).

## 4. Applications

The probe is trained once, offline. At inference it costs ~1μs. All three applications below use the same trained probe — no extra labels, no retraining.

![Inference-time routing cascade: cost accounting and three-tier decision logic](assets/inference_cascade.png)

### Out-of-distribution detection

CIFAR-10 (in-distribution) vs SVHN (OOD) using the same ViT-MAE probe:

| Method | AUROC | Compute cost |
|--------|-------|--------------|
| Random baseline | 0.51 | — |
| **Ridge probe (this work)** | **0.88** | **~1% of oracle** |
| Full decoder oracle | 0.96 | 100% |

88% AUROC at 1% of the cost of running the decoder.

### Data quality filtering

Filtering the top-15% highest-difficulty training samples before retraining ResNet-18 on noisy CIFAR-10:

| Label noise | Baseline | Filtered | Δ |
|-------------|----------|----------|---|
| 0% | 89.3% | 91.2% | +1.9% |
| 40% | 74.6% | 77.0% | **+2.4%** |

### Active learning

At 200 labeled samples (50 initial + 3 rounds of 50):

| Strategy | Accuracy | Δ vs random |
|----------|----------|-------------|
| Random | 12.1% | — |
| Ridge probe uncertainty | 13.3% | **+1.2%** |
| MC-Dropout (10 passes) | 14.4% | +2.4% |

MC-Dropout wins here, but it runs 10× more forward passes per candidate. The probe selects from large unlabeled pools at essentially zero marginal cost.

## 5. Open questions

- **Why is the signal linear?** The probe is Ridge regression — a linear model. Why doesn't a deeper probe do better? The dimension ablation shows random subsets of dimensions perform as well as ordered prefixes, suggesting the signal is distributed uniformly through the embedding. Why does the training process deposit difficulty in such a linearly accessible form?

- **Connection to JEPA-SCORE.** [Van Assel & Balestriero (2025)](https://arxiv.org/html/2510.05949) independently showed that *contrastive* models (JEPA, DINO) encode data density in their representations via the Jacobian. We see the same thing in *reconstruction* models via a linear probe. Is there a unified theory across all SSL paradigms?

- **Can we use this for hallucination reduction?** The original motivation was a composite score: `log_prob(generation) - λ × predicted_loss(CLS)`. A high predicted reconstruction loss flags that the input is atypical — and atypical inputs are where generative models hallucinate. We haven't tested this end-to-end.

- **Class vs instance loss decomposition.** The scalar loss bundles two independent signals (class-level difficulty and instance-level atypicality, r≈0.000 between them). 91% of variance is instance-level. Can we separate them without labels in multi-semantic geospatial scenes (trail vs stream = class confusion; unusual cloud = instance atypicality)? Requires labeled geospatial data — a concrete next experiment.

- **Spatial difficulty.** The CLS encodes *global* difficulty. Patch-level Spearman r = 0.097 — the signal doesn't localize. Running the decoder does. Is there an intermediate approach that localizes without full decoder cost?

## Citation

```bibtex
@misc{lgnd2026selfaware,
  title   = {Self-Aware Embeddings: How Foundation Models Encode Their Own Ignorance},
  author  = {S\'anchez-Andrade Nu\~no, Bruno},
  year    = {2026},
  month   = {3},
  url     = {https://blog.lgnd.ai/posts/2026-03-01-self-aware-embeddings/},
  note    = {LGND Science technical post}
}
```

Full technical memo and code: [github.com/EarthLegend/loss-regresson-experiment](https://github.com/EarthLegend/loss-regresson-experiment)